# 08 · Evaluation

Same meetings, same references, same metric implementation for every system.

- Summarisation: ROUGE-1/2/L (`rouge_score`), optional BERTScore (`--with-bertscore`)
- Classification: precision / recall / F1, confusion matrix (vs **weak** labels — labelled as such)
- ASR: WER (`jiwer`, only with a reference transcript)
- Action extraction: P/R/F1 + owner/deadline exact match where a reference exists

Results are written only to `outputs/metrics/results.{csv,json}` — the reports quote
**nothing else**.

In [ ]:
print('[1/4] Run the controlled experiment (smoke size = 5 meetings)')
!python -u scripts/evaluate.py --split val --limit 5 --with-lstm

In [ ]:
print('[2/4] Load results')
import sys; sys.path.insert(0, '.')
import pandas as pd
from src.config import load_config, resolve_path
cfg = load_config()
df = pd.read_json(resolve_path(cfg, 'metrics') / 'results.json')
cols = [c for c in ['system', 'n_meetings', 'rouge1', 'rouge2', 'rougeL',
                    'bertscore_f1', 'runtime_seconds'] if c in df.columns]
print(df[cols].to_string(index=False))

In [ ]:
print('[3/4] ROUGE comparison chart')
import matplotlib.pyplot as plt
r = df[df['rouge1'].notna()]
x = range(len(r))
w = 0.27
plt.figure(figsize=(8, 4))
for i, (k, lab) in enumerate([('rouge1', 'R-1'), ('rouge2', 'R-2'), ('rougeL', 'R-L')]):
    plt.bar([xi + i * w for xi in x], r[k], w, label=lab)
plt.xticks([xi + w for xi in x], r['system'], rotation=20, ha='right')
plt.ylim(0, 1); plt.ylabel('F1'); plt.legend(); plt.title('ROUGE on the same meetings')
plt.tight_layout(); plt.show()

In [ ]:
print('[4/4] Generate all figures from real metric files')
!python -u scripts/make_figures.py
from pathlib import Path
for p in sorted(Path('outputs/figures').glob('*.png')):
    print(' ', p)